# Train 500M Legal-Finance-Accounting LLM on Free T4

Trains a ~500M parameter LLaMA model from scratch on legal, finance, accounting & auditing text.

**Data Mix**:
- Legal (30%): NVIDIA Nemotron-Legal (California Code, Case Law, eCFR, NYC Ethics, CaseHOLD, GlobalCit)
- Finance/Accounting (50%): SEC financial reports, PCAOB standards, IFRS/US GAAP
- General (20%): FineWeb-Edu

**Architecture**: 528M param LLaMA (hidden=1152, layers=24, heads=18, vocab=16384)

## 1. Install Dependencies

In [ ]:
!pip install -q transformers datasets tokenizers accelerate huggingface_hub

In [ ]:
from huggingface_hub import login
import getpass
token = getpass.getpass("HF token: ")
login(token, add_to_git_credential=False)
print("Logged in!")

## 2. Load Domain Data

Loads from three domains: **Legal**, **Finance**, and **General** text.

All datasets use streaming to avoid disk overflow.

In [ ]:
from datasets import load_dataset, concatenate_datasets
import itertools

MAX_PER_SOURCE = 100000
MAX_TOTAL = 300000

train_texts = []

# --- LEGAL (30%) ---
print("=== Loading Legal Data ===")
legal_configs = [
    "Nemotron-Pretraining-Legal-California-Code-Of-Regulations",
    "Nemotron-Pretraining-Legal-Case-Law-Summary",
    "Nemotron-Pretraining-Legal-eCFR",
    "Nemotron-Pretraining-Legal-NYCourts-Judicial-Ethics-Opinions",
    "Nemotron-Pretraining-Legal-GlobalCit",
    "Nemotron-Pretraining-Legal-CaseHOLD",
]
for cfg in legal_configs:
    print(f"  Loading Nemotron {cfg}...")
    try:
        ds = load_dataset("nvidia/Nemotron-Pretraining-Legal-v1", cfg,
                          split="train", streaming=True)
        count = 0
        for x in ds:
            if count >= MAX_PER_SOURCE:
                break
            if len(train_texts) >= MAX_TOTAL:
                break
            for field in ["text", "input", "question", "answer", "content"]:
                if field in x and x[field] and isinstance(x[field], str):
                    train_texts.append(x[field][:2048])
                    count += 1
                    break
        print(f"    -> {count} examples loaded ({len(train_texts)} total)")
    except Exception as e:
        print(f"    Skip: {e}")
    if len(train_texts) >= MAX_TOTAL:
        break

print(f"\nLegal done. Total: {len(train_texts)}")

In [ ]:
# --- FINANCE / ACCOUNTING (50%) ---
print("=== Loading Finance Data ===")

# SEC financial reports from HF
sec_count = 0
try:
    sec_ds = load_dataset("JanosAudran/financial-reports-sec", split="train",
                          streaming=True)
    for x in sec_ds:
        if len(train_texts) >= MAX_TOTAL or sec_count >= MAX_PER_SOURCE:
            break
        if "text" in x and x["text"] and isinstance(x["text"], str):
            train_texts.append(x["text"][:2048])
            sec_count += 1
except Exception as e:
    print(f"  SEC dataset error: {e}")
print(f"  SEC reports: {sec_count} loaded ({len(train_texts)} total)")

# Also load additional finance text from Investopedia / financial glossary
fin_count = 0
try:
    fin_ds = load_dataset("infCapital/investopedia_terms_en", split="train",
                          streaming=True)
    for x in fin_ds:
        if len(train_texts) >= MAX_TOTAL or fin_count >= 50000:
            break
        if "text" in x and x["text"] and isinstance(x["text"], str):
            train_texts.append(x["text"][:2048])
            fin_count += 1
except Exception as e:
    print(f"  Investopedia error: {e}")
print(f"  Investopedia: {fin_count} loaded ({len(train_texts)} total)")

print(f"\nFinance done. Total: {len(train_texts)}")

In [ ]:
# --- GENERAL (20%) ---
print("=== Loading General Data ===")
gen_count = 0
try:
    gen_ds = load_dataset("HuggingFaceFW/fineweb-edu", "sample-10BT",
                          split="train", streaming=True)
    for x in gen_ds:
        if len(train_texts) >= MAX_TOTAL or gen_count >= 60000:
            break
        if "text" in x and x["text"] and isinstance(x["text"], str):
            train_texts.append(x["text"][:2048])
            gen_count += 1
except Exception as e:
    print(f"  FineWeb-Edu error: {e}")
    try:
        gen_ds = load_dataset("HuggingFaceFW/fineweb", "sample-10BT",
                              split="train", streaming=True)
        for x in gen_ds:
            if len(train_texts) >= MAX_TOTAL or gen_count >= 60000:
                break
            if "text" in x and x["text"] and isinstance(x["text"], str):
                train_texts.append(x["text"][:2048])
                gen_count += 1
    except Exception as e2:
        print(f"  FineWeb error: {e2}")
print(f"  General: {gen_count} loaded")

print(f"\n=== TOTAL EXAMPLES: {len(train_texts)} ===")

## 3. Train Domain Tokenizer (vocab=16384)

Trains a BPE tokenizer on our domain text. Small vocab = more efficient for specialized domain.

In [ ]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders
from transformers import PreTrainedTokenizerFast

tokenizer = Tokenizer(models.BPE(unk_token="<unk>"))
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
tokenizer.decoder = decoders.ByteLevel()

trainer = trainers.BpeTrainer(
    vocab_size=16384,
    special_tokens=["<unk>", "<s>", "</s>", "<pad>"],
    min_frequency=2,
)
tokenizer.train_from_iterator(train_texts, trainer)

hf_tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=tokenizer,
    unk_token="<unk>",
    bos_token="<s>",
    eos_token="</s>",
    pad_token="<pad>",
)
print(f"Vocab size: {hf_tokenizer.vocab_size}")
print(f"Sample: {hf_tokenizer.decode(hf_tokenizer.encode('The court finds that the defendant'))}")

## 4. Create Model (~528M params)

LLaMA architecture with grouped-query attention for efficiency.

**Architecture**:
- Vocab: 16,384
- Hidden: 1,152
- Layers: 24
- Heads: 18
- Intermediate: 4,608 (4x hidden)
- Params: ~528M

In [ ]:
import torch
from transformers import LlamaConfig, LlamaForCausalLM

config = LlamaConfig(
    vocab_size=hf_tokenizer.vocab_size,
    hidden_size=1152,
    intermediate_size=4608,
    num_hidden_layers=24,
    num_attention_heads=18,
    max_position_embeddings=1024,
    rope_theta=10000.0,
    tie_word_embeddings=True,
    use_cache=True,
    torch_dtype="bfloat16",
)

model = LlamaForCausalLM(config)
total = sum(p.numel() for p in model.parameters())
print(f"Model: {total:,} params ({total/1e6:.1f}M)")
print(f"VRAM est (fp16): {total * 2 / 1e9:.2f}GB")
print(f"VRAM est (states): {total * 12 / 1e9:.2f}GB (weights+grad+optim)")

## 5. Tokenize Dataset

In [ ]:
from datasets import Dataset

MAX_LENGTH = 512

def encode(texts):
    return hf_tokenizer(texts, truncation=True, max_length=MAX_LENGTH,
                        padding=False)["input_ids"]

# Shuffle texts deterministically
import random
random.seed(42)
random.shuffle(train_texts)

dataset = Dataset.from_dict({"text": train_texts})
dataset = dataset.map(
    lambda x: {"input_ids": encode(x["text"])},
    remove_columns=["text"],
    desc="Tokenizing",
)
dataset = dataset.filter(
    lambda x: len(x["input_ids"]) > 10,
    desc="Filtering short",
)
print(f"Examples: {len(dataset)}")

## 6. Data Collator

In [ ]:
from transformers import DataCollatorForLanguageModeling

collator = DataCollatorForLanguageModeling(
    tokenizer=hf_tokenizer, mlm=False, pad_to_multiple_of=8
)

## 7. Train (~6-10 hours on T4)

**Settings**:
- Batch: 2 per device, 16 grad accum = 32 effective
- Precision: fp16 mixed
- Grad checkpointing: ON
- LR: 3e-4 cosine
- Epochs: 10 (until plateau)

In [ ]:
from transformers import TrainingArguments, Trainer, TrainerCallback
from huggingface_hub import HfApi

MODEL_NAME = "pink-elephant-legalfinance-500m"
REPO_ID = f"pinkelephantlimited/{MODEL_NAME}"

# Ensure repo exists
HfApi().create_repo(REPO_ID, private=False, repo_type="model", exist_ok=True)
print(f"Repo {REPO_ID} ready")

class HFSaveCallback(TrainerCallback):
    def on_save(self, args, state, control, **kwargs):
        ckpt_dir = f"{args.output_dir}/checkpoint-{state.global_step}"
        if os.path.exists(ckpt_dir):
            print(f"\nUploading checkpoint-{state.global_step} to HF...")
            HfApi().upload_folder(
                folder_path=ckpt_dir,
                repo_id=REPO_ID,
                path_in_repo=f"checkpoints/checkpoint-{state.global_step}",
                ignore_patterns=["*.bin", "optimizer.pt", "scheduler.pt", "rng_state.pth"],
            )
            print(f"  -> Uploaded")

args = TrainingArguments(
    output_dir="./" + MODEL_NAME,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=16,
    num_train_epochs=10,
    learning_rate=3e-4,
    weight_decay=0.01,
    warmup_steps=500,
    logging_steps=10,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,
    report_to="none",
    fp16=True,
    optim="adamw_torch",
    dataloader_num_workers=2,
    remove_unused_columns=False,
    gradient_checkpointing=True,
    ddp_find_unused_parameters=False,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=dataset,
    data_collator=collator,
    callbacks=[HFSaveCallback],
)
trainer.train()

## 8. Quick Generation Test

In [ ]:
prompts = [
    "The court finds that",
    "Pursuant to Section 5 of the Act,",
    "In accordance with IFRS standards,",
    "The auditor shall",
    "Under PCAOB standards,",
    "The company's financial statements",
]
for p in prompts:
    inputs = hf_tokenizer(p, return_tensors="pt")
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=80, do_sample=True,
            temperature=0.5, top_p=0.9,
            pad_token_id=hf_tokenizer.pad_token_id,
        )
    print(f"\n{p}\n  -> {hf_tokenizer.decode(out[0], skip_special_tokens=True)}")

## 9. Save & Upload to Hugging Face

In [ ]:
import os, json, shutil
from huggingface_hub import HfApi

save_dir = "/tmp/" + MODEL_NAME
if os.path.exists(save_dir):
    shutil.rmtree(save_dir)

model.save_pretrained(save_dir, safe_serialization=True)
hf_tokenizer.save_pretrained(save_dir)

# Fix config
cfg_path = os.path.join(save_dir, "config.json")
with open(cfg_path) as f:
    cfg = json.load(f)
cfg["_name_or_path"] = f"pinkelephantlimited/{MODEL_NAME}"
cfg["architectures"] = ["LlamaForCausalLM"]
with open(cfg_path, "w") as f:
    json.dump(cfg, f, indent=2)

# License
license = """MIT License

Copyright (c) 2026 Pink Elephant Limited

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE."""
with open(os.path.join(save_dir, "LICENSE"), "w") as f:
    f.write(license)

# Upload
api = HfApi()
repo_id = f"pinkelephantlimited/{MODEL_NAME}"
api.create_repo(repo_id, private=False, repo_type="model", exist_ok=True)
api.upload_folder(folder_path=save_dir, repo_id=repo_id, ignore_patterns=["*.bin"])
print(f"Uploaded to: https://huggingface.co/{repo_id}")

In [ ]:
print("DONE! Model is live at:")
print(f"https://huggingface.co/pinkelephantlimited/{MODEL_NAME}")